# Fechamento da contribuição C

Três análises que faltavam para fechar a contribuição C:
1. **Controles tóxicos por categoria emocional:** por que 37% dos controles já saem como tóxicos no BERTimbau toxic com o limiar 0,5.
2. **RQ5:** para cada termo, quantas vezes ele aparece nos dados de treino de cada modelo, ao lado do efeito dele na v1.1.
3. **Visto e não visto:** desempenho de cada modelo em dados que ele viu no treino, comparado com os corpora que ele não viu.

**Entrada:** `data_bits/BITS_PTBR_v1.1_pareado_com_scores.csv`, `lista_termos_congelada.csv` e, em `results`, `thresholds.json`, `c1_contagem_termos_por_corpus.csv`, `c1_inspecao_corpora.csv`, `b_tabela_por_termo_v1.1.csv` (notebook 04), `c4_sensibilidade_limiar_v1.1_por_termo.csv` e `calibration_metrics.csv`.

**Saída (em `results`):** `c5_controles_toxicos_por_categoria.csv`, `c6_rq5_termos_treino_x_efeito.csv`, `c7_visto_vs_nao_visto.csv` e os scores novos (`olid_br_scores_tox1.csv` e `told_br_validacao_vistos_scores_tox2.csv`).



In [2]:
from google.colab import drive
drive.mount('/content/drive')

import ast, json, re, unicodedata
import numpy as np, pandas as pd

ROOT = "/content/drive/MyDrive/JIS - capacitismo"
BASE = f"{ROOT}/data_externa"
OUT  = f"{ROOT}/results"

with open(f"{OUT}/thresholds.json") as f:
    thresholds = json.load(f)
v11 = pd.read_csv(f"{ROOT}/data_bits/BITS_PTBR_v1.1_pareado_com_scores.csv")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("v1.1:", v11.shape, "| limiares:", thresholds)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
v1.1: (2730, 22) | limiares: {'tox1': {'padrao_0.5': 0.5, 'youden': 0.7966679930686951, 'fpr_5': 0.8413420140743255}, 'tox2': {'padrao_0.5': 0.5, 'youden': 0.4324200749397278, 'fpr_5': 0.8765263080596923}}


## 1. Controles tóxicos por categoria emocional

Usa as 130 frases de controle distintas da v1.1, que são as mesmas da v1. Para cada limiar (0,5, Youden e 5% de falsos positivos), calcula a fração de controles que o modelo marca como tóxicos, por categoria emocional e por template.

In [3]:
ctrl = v11.drop_duplicates("Sentence_control")[["Template", "emocao_categoria", "Sentence_control",
                                                 "tox1_score_control", "tox2_score_control"]].reset_index(drop=True)
print("controles distintos:", len(ctrl))

MODELS = {"tox1": "BERTimbau toxic", "tox2": "ToxiGuardrail-PT"}
rows = []
for key, model in MODELS.items():
    for thr_name, thr in thresholds[key].items():
        toxic = ctrl[f"{key}_score_control"] >= thr
        for by in ["emocao_categoria", "Template"]:
            for value, g in toxic.groupby(ctrl[by]):
                rows.append({"modelo": model, "limiar": thr_name, "recorte": by, "valor": value,
                             "controles": len(g), "toxicos": int(g.sum()), "taxa": g.mean()})
        rows.append({"modelo": model, "limiar": thr_name, "recorte": "total", "valor": "todos",
                     "controles": len(toxic), "toxicos": int(toxic.sum()), "taxa": toxic.mean()})
controles = pd.DataFrame(rows)
controles.to_csv(f"{OUT}/c5_controles_toxicos_por_categoria.csv", index=False)

ORDEM = ["padrao_0.5", "youden", "fpr_5"]
t1 = controles[controles.modelo == "BERTimbau toxic"]
print("\nBERTimbau toxic, total:")
print(t1[t1.recorte == "total"].set_index("limiar").loc[ORDEM, ["controles", "toxicos", "taxa"]].round(3))
print("\nBERTimbau toxic, por categoria emocional (taxa):")
print(t1[t1.recorte == "emocao_categoria"].pivot(index="valor", columns="limiar", values="taxa")[ORDEM]
      .sort_values("padrao_0.5", ascending=False).round(3))
print("\nBERTimbau toxic, por template (taxa):")
print(t1[t1.recorte == "Template"].pivot(index="valor", columns="limiar", values="taxa")[ORDEM].round(3))
print("\nToxiGuardrail-PT, controles tóxicos em cada limiar:",
      controles[(controles.modelo == "ToxiGuardrail-PT") & (controles.recorte == "total")].set_index("limiar").toxicos.to_dict())

neutros = ctrl[(ctrl.emocao_categoria == "neutro") & (ctrl.tox1_score_control >= 0.5)]
print("\ncontroles neutros tóxicos no BERTimbau toxic com 0,5:")
print(neutros[["Sentence_control", "tox1_score_control"]].to_string(index=False))

controles distintos: 130

BERTimbau toxic, total:
            controles  toxicos   taxa
limiar                               
padrao_0.5        130       48  0.369
youden            130       13  0.100
fpr_5             130        5  0.038

BERTimbau toxic, por categoria emocional (taxa):
limiar        padrao_0.5  youden  fpr_5
valor                                  
nojo               0.933   0.400  0.133
raiva              0.933   0.133  0.133
medo               0.467   0.067  0.000
tristeza           0.400   0.133  0.067
surpresa_neg       0.333   0.167  0.000
felicidade         0.077   0.000  0.000
surpresa_pos       0.067   0.000  0.000
neutro             0.033   0.000  0.000

BERTimbau toxic, por template (taxa):
limiar  padrao_0.5  youden  fpr_5
valor                            
T1            0.00    0.00    0.0
T10           0.60    0.36    0.2
T2            0.00    0.00    0.0
T3            0.00    0.00    0.0
T4            0.00    0.00    0.0
T5            0.00    0.00    0.0

## 2. RQ5: o termo nos dados de treino e o efeito dele

Para cada termo de deficiência da lista congelada:
- **No treino:** quantos textos com o termo estão do lado nocivo e do lado não nocivo. Para o ToxiGuardrail-PT, o treino é o `harmful-text`. Para o BERTimbau toxic, é o OLID-BR.
- **Proporção nociva:** só é calculada quando o termo aparece em pelo menos 10 textos. Deve ser comparada com a proporção do corpus inteiro (`prop_base`): 50% no `harmful-text` e 85% no OLID-BR.
- **Efeito na v1.1:** o delta médio e o intervalo de 95%, da tabela por termo do notebook 04, e a fração de pares que passam de não tóxico a tóxico no limiar de 5% de falsos positivos, da tabela de sensibilidade ao limiar.

Os termos que não estão na v1.1 (os quatro acréscimos e "portador de deficiência") aparecem sem efeito. A contagem deles serve como previsão para a v2.

A comparação é descritiva: com tão poucos termos, não cabe calcular correlação.

In [4]:
termos   = pd.read_csv(f"{ROOT}/lista_termos_congelada.csv")
contagem = pd.read_csv(f"{OUT}/c1_contagem_termos_por_corpus.csv")
inspecao = pd.read_csv(f"{OUT}/c1_inspecao_corpora.csv")
tabela   = pd.read_csv(f"{OUT}/b_tabela_por_termo_v1.1.csv")
flips    = pd.read_csv(f"{OUT}/c4_sensibilidade_limiar_v1.1_por_termo.csv")

tabela["modelo_n"] = tabela.modelo.str.lower()
flips["modelo_n"] = flips.modelo.str.lower()

def counts(corpus, nocivo, nao_nocivo):
    c = (contagem[contagem.corpus == corpus]
         .pivot_table(index="termo", columns="rotulo", values="textos", aggfunc="sum")
         .reindex(columns=[nocivo, nao_nocivo]).fillna(0).astype(int))
    c.columns = ["nocivo", "nao_nocivo"]
    return c

def base_rate(corpus, nocivo):
    r = ast.literal_eval(inspecao.set_index("corpus").loc[corpus, "rotulos"])
    return r[nocivo] / sum(r.values())

MIN_TEXTOS = 10
TREINOS = {"harmful_text": ("harmful-text (PT)", "harmful", "harmless"),
           "olid":         ("OLID-BR", "ofensivo", "não ofensivo")}

alvo = termos[termos.grupo != "Só E (xingamento)"].copy()
for key, (corpus, nocivo, nao_nocivo) in TREINOS.items():
    c = counts(corpus, nocivo, nao_nocivo).reindex(alvo.termo).fillna(0).astype(int)
    total = c.nocivo + c.nao_nocivo
    alvo[f"{key}_nocivo"] = c.nocivo.values
    alvo[f"{key}_nao_nocivo"] = c.nao_nocivo.values
    alvo[f"{key}_prop_nocivo"] = np.where(total.values >= MIN_TEXTOS, (c.nocivo / total.replace(0, np.nan)).values, np.nan)
    alvo[f"{key}_prop_base"] = base_rate(corpus, nocivo)

for model, prefix in [("toxiguardrail-pt", "toxiguardrail"), ("bertimbau toxic", "bertimbau")]:
    t = tabela[tabela.modelo_n == model].set_index("SubClass")[["score_sense", "ic95_inf", "ic95_sup"]]
    f = flips[(flips.modelo_n == model) & (flips.limiar == "fpr_5")].set_index("SubClass").flip_para_toxico
    alvo[f"{prefix}_score_sense"] = alvo.subclass_v1.map(t.score_sense)
    alvo[f"{prefix}_ic95_inf"] = alvo.subclass_v1.map(t.ic95_inf)
    alvo[f"{prefix}_ic95_sup"] = alvo.subclass_v1.map(t.ic95_sup)
    alvo[f"{prefix}_flip_para_toxico_fpr5"] = alvo.subclass_v1.map(f)

rq5 = alvo.drop(columns=["padrao_busca", "data_congelamento"])
rq5.to_csv(f"{OUT}/c6_rq5_termos_treino_x_efeito.csv", index=False)

print("Base: harmful-text", round(base_rate("harmful-text (PT)", "harmful"), 3), "| OLID-BR", round(base_rate("OLID-BR", "ofensivo"), 3))
print("\nToxiGuardrail-PT: termo no treino (harmful-text) x efeito na v1.1")
print(rq5[["termo", "harmful_text_nocivo", "harmful_text_nao_nocivo", "harmful_text_prop_nocivo",
           "toxiguardrail_score_sense", "toxiguardrail_ic95_inf", "toxiguardrail_ic95_sup",
           "toxiguardrail_flip_para_toxico_fpr5"]].round(3).to_string(index=False))
print("\nBERTimbau toxic: termo no treino (OLID-BR) x efeito na v1.1")
print(rq5[["termo", "olid_nocivo", "olid_nao_nocivo", "olid_prop_nocivo",
           "bertimbau_score_sense", "bertimbau_ic95_inf", "bertimbau_ic95_sup",
           "bertimbau_flip_para_toxico_fpr5"]].round(3).to_string(index=False))

Base: harmful-text 0.5 | OLID-BR 0.854

ToxiGuardrail-PT: termo no treino (harmful-text) x efeito na v1.1
                               termo  harmful_text_nocivo  harmful_text_nao_nocivo  harmful_text_prop_nocivo  toxiguardrail_score_sense  toxiguardrail_ic95_inf  toxiguardrail_ic95_sup  toxiguardrail_flip_para_toxico_fpr5
  com transtorno do espectro autista                    0                        0                       NaN                      0.032                   0.023                   0.041                                0.000
com transtorno de déficit de atenção                    1                        0                       NaN                     -0.012                  -0.021                  -0.004                                0.000
              com paralisia cerebral                    0                        1                       NaN                      0.011                   0.004                   0.018                                0.000
          

## 3. Desempenho em dados vistos e não vistos (precisa de GPU)

- **BERTimbau toxic:** é rodado no OLID-BR, separando treino (dado de ajuste do modelo) e teste (não se sabe se foi usado no ajuste).
- **ToxiGuardrail-PT:** é rodado nos tuítes da validação do ToLD-Br que estão no `harmful-text`, ou seja, que ele viu no treino. Como o teste do ToLD-Br é do mesmo corpus e não foi visto, a comparação isola o efeito de ter visto o texto.

Os resultados nos corpora não vistos (metade B do HateBR e teste do ToLD-Br) vêm do `calibration_metrics.csv`, já calculado.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", device)

@torch.no_grad()
def score_texts(model_name, kind, texts, batch_size=32, max_length=512):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()
    if kind == "bertimbau":
        off_id = {v: k for k, v in model.config.id2label.items()}["OFFENSIVE"]
    scores = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], padding=True, truncation=True,
                  max_length=max_length, return_tensors="pt").to(device)
        logits = model(**enc).logits
        if kind == "bertimbau":
            s = torch.softmax(logits, dim=-1)[:, off_id]
        else:
            s = 1 - torch.sigmoid(logits.squeeze(-1))
        scores.extend(s.float().cpu().tolist())
    del model
    torch.cuda.empty_cache()
    return np.array(scores)

def norm(s):
    s = str(s).lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'^rt\b', ' ', s)
    s = re.sub(r'\buser\b', ' ', s)
    s = unicodedata.normalize('NFKC', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

olid = pd.concat([pd.read_parquet(f"{BASE}/olid_br_train.parquet").assign(split="treino"),
                  pd.read_parquet(f"{BASE}/olid_br_test.parquet").assign(split="teste")], ignore_index=True)
olid["label"] = (olid.is_offensive == "OFF").astype(int)
olid["tox1_score"] = score_texts("dougtrajano/toxic-comment-classification", "bertimbau", olid.text.astype(str).tolist())
olid[["split", "text", "label", "tox1_score"]].to_csv(f"{OUT}/olid_br_scores_tox1.csv", index=False)

ht = pd.read_parquet(f"{BASE}/harmful_text_pt_train.parquet")
seen = set(ht.harmless.map(norm)) | set(ht.harmful.map(norm))
told_val = pd.read_csv(f"{BASE}/told_br/1annotator/ptbr_validation_1annotator.csv").rename(columns={"toxic": "label"})
told_val = told_val[told_val.text.map(norm).isin(seen)].reset_index(drop=True)
told_val["tox2_score"] = score_texts("nicholasKluge/ToxiGuardrailPT", "guardrail", told_val.text.astype(str).tolist())
told_val.to_csv(f"{OUT}/told_br_validacao_vistos_scores_tox2.csv", index=False)
print("OLID-BR:", len(olid), "textos | validação do ToLD-Br vista pelo ToxiGuardrail-PT:", len(told_val), "tuítes")

dispositivo: cuda


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

OLID-BR: 6952 textos | validação do ToLD-Br vista pelo ToxiGuardrail-PT: 1459 tuítes


In [6]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

def evaluate(y, s, thr):
    pred = (s >= thr).astype(int)
    return {"acuracia": accuracy_score(y, pred), "f1_macro": f1_score(y, pred, average="macro"),
            "f1_ofensivo": f1_score(y, pred), "taxa_falso_positivo": float(pred[y == 0].mean()),
            "sensibilidade": float(pred[y == 1].mean())}

rows = []
vistos = [("tox1", "OLID-BR treino", "visto (dado de ajuste)", olid[olid.split == "treino"]),
          ("tox1", "OLID-BR teste", "não se sabe se foi usado no ajuste", olid[olid.split == "teste"]),
          ("tox2", "ToLD-Br validação (tuítes do treino)", "visto (no harmful-text)", told_val)]
for key, corpus, status, d in vistos:
    y, s = d.label.values, d[f"{key}_score"].values
    for thr_name, thr in thresholds[key].items():
        rows.append({"modelo": key, "corpus": corpus, "status": status, "n": len(d), "limiar": thr_name,
                     "auc_roc": roc_auc_score(y, s), **evaluate(y, s, thr)})

cal = pd.read_csv(f"{OUT}/calibration_metrics.csv")
cal["status"] = "não visto"
vvn = pd.concat([pd.DataFrame(rows), cal.drop(columns=["valor_limiar", "precisao_media"])], ignore_index=True)
vvn["modelo"] = vvn.modelo.map(MODELS)
vvn.to_csv(f"{OUT}/c7_visto_vs_nao_visto.csv", index=False)

cols = ["modelo", "corpus", "status", "auc_roc", "acuracia", "f1_macro", "taxa_falso_positivo", "sensibilidade"]
print("Com o limiar de 0,5:")
print(vvn[vvn.limiar == "padrao_0.5"][cols].round(3).to_string(index=False))
print("\nCom o limiar de 5% de falsos positivos:")
print(vvn[vvn.limiar == "fpr_5"][cols].round(3).to_string(index=False))

Com o limiar de 0,5:
          modelo                               corpus                             status  auc_roc  acuracia  f1_macro  taxa_falso_positivo  sensibilidade
 BERTimbau toxic                       OLID-BR treino             visto (dado de ajuste)    0.785     0.842     0.604                0.770          0.946
 BERTimbau toxic                        OLID-BR teste não se sabe se foi usado no ajuste    0.768     0.841     0.616                0.740          0.940
ToxiGuardrail-PT ToLD-Br validação (tuítes do treino)            visto (no harmful-text)    0.893     0.769     0.758                0.434          0.966
 BERTimbau toxic                      HateBR metade B                          não visto    0.932     0.827     0.825                0.272          0.925
 BERTimbau toxic              ToLD-Br teste nao visto                          não visto    0.807     0.731     0.721                0.155          0.595
ToxiGuardrail-PT                      HateBR metade B  

## Leitura dos resultados

Conferir os números com a saída das células.

**Parte 1: controles tóxicos**
- No BERTimbau toxic, 48 dos 130 controles (36,9%) saem como tóxicos com 0,5; 13 (10,0%) com o limiar de Youden; 5 (3,8%) com o de 5% de falsos positivos. No ToxiGuardrail-PT, nenhum, em nenhum limiar.
- Com 0,5, a taxa é de 93% em nojo e em raiva, de 33% a 47% em medo, tristeza e surpresa negativa, e de 3% a 8% em felicidade, surpresa positiva e neutro. Nenhum controle de T1 a T5 sai como tóxico.
- O score de base vem da palavra emocional, e não de uma falha geral do modelo. O único controle neutro tóxico é "A pessoa estava em uma situação esperado", frase sem concordância, e o T10, que tem esse problema, é o template com a maior taxa (60%).

**Parte 2: RQ5**
- No treino do ToxiGuardrail-PT, "cego" (98% nocivo) e "surdo" (99,5%) aparecem quase só em textos nocivos e têm efeito médio claro (0,142 e 0,179). "Pessoa com deficiência" aparece dos dois lados (56% nocivo) e tem efeito pequeno (0,020). Até aqui, a contagem acompanha o efeito.
- Mas "cadeirante" não aparece nenhuma vez no treino e tem o maior efeito (0,213). E, no limiar de 5% de falsos positivos, "cego" quase não muda a decisão (1,5% dos pares), enquanto "cadeirante" muda 16%. A associação com os dados de treino é, portanto, parcial.
- No OLID-BR, os termos de deficiência são raros. "Cego" aparece em 19 textos, 74% ofensivos, abaixo dos 85% do corpus inteiro, e mesmo assim é o termo de maior efeito no BERTimbau toxic (0,429). Os dados de ajuste do BERTimbau toxic não explicam o efeito dele.
- Previsão para a v2: "deficiente" (99% nocivo no treino do ToxiGuardrail-PT) e "deficiente mental" (100%) devem ter efeito alto nesse modelo, se a associação com o treino se confirmar.

**Parte 3: visto e não visto**
- Ter visto o texto no treino dá pouca ou nenhuma vantagem. O BERTimbau toxic tem AUC de 0,785 no treino do OLID-BR e de 0,768 no teste. O ToxiGuardrail-PT tem AUC de 0,893 nos tuítes da validação do ToLD-Br que estão no seu treino e de 0,856 no teste, que não viu, com taxa de falso positivo parecida nos dois (43,4% e 40,3% com 0,5).
- O BERTimbau toxic discrimina pior no próprio dado de ajuste (AUC de 0,785 no OLID-BR) do que no HateBR, que não viu (0,932). Com 0,5, marca como ofensivos 77% dos textos não ofensivos do OLID-BR e 92% de todos os textos, contra 85% de ofensivos reais. O modelo tende a marcar quase tudo como ofensivo, o que é coerente com os 37% de controles tóxicos na v1.1.
- No teste do OLID-BR, com 0,5, a acurácia é 0,841 e o F1 ponderado 0,824. A ficha do modelo reporta 0,855 e 0,855. A diferença é pequena e é compatível com o uso de outra versão do OLID-BR no ajuste.
- Consequências: o vazamento de treino não compromete a validação nos corpora não vistos. E a associação entre os dados de treino e o efeito dos termos (RQ5), onde existe, não é memorização de textos, mas frequência com que o termo aparece em textos nocivos.